# Entrenament model YOLO 2D + seguiment granja final

## Entrenament YOLOv8m

In [ ]:
from ultralytics import YOLO

# configuració de rutes
# apuntem al dataset que acabem de dividir
ruta_dataset_yaml = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/Dataset_nova_granja/Dataset_groun_Truth_3d/dataset_yolo_combined/data.yaml'
ruta_guardat_projecte = '/content/drive/MyDrive/Tfg/comparativa_models'

# carreguem el model base
print("Descarregant el model YOLOv8 medium...")
model_medium = YOLO('yolov8m-seg.pt')

# entrenament
print("Iniciant l'entrenament del model...")

resultats = model_medium.train(
    data=ruta_dataset_yaml,
    epochs=100,
    imgsz=640,
    batch=8,              # per no saturar la GPU de Colab
    device=0,
    patience=20,          # parem si no millora en 20 èpoques
    project=ruta_guardat_projecte,
    name='yolov8m_definitiu_220imgs', # nom per no xafar models antics
    exist_ok=True
)

print("Entrenament completat.")
print(f"Els pesos 'best.pt' s'han guardat a: {ruta_guardat_projecte}/yolov8m_definitiu_220imgs/weights/")


## Creació dataset autoencoder

In [ ]:
import cv2
import os
import numpy as np
from ultralytics import YOLO

# rutes i configuració
# ruta dels nous pesos acabats d'entrenar
ruta_yolo_nou = '/content/drive/MyDrive/Tfg/comparativa_models/yolov8m_definitiu_220imgs/weights/best.pt'

# rutes dels vídeos d'entrada
rutes_videos_in = [
    '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/Dataset_nova_granja/Camera2D/Camera2D/room1_cam2_2026-04-19_08-05-10_retallat.mp4',
    '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/Dataset_nova_granja/Camera2D/Camera2D/room1_cam1_2026-04-19_08-00-12.ts',
    '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/Dataset_nova_granja/Camera2D/Camera2D/room1_cam2_2026-04-19_13-00-10_retallat.mp4'
]

# on guardarem les imatges del dataset
ruta_dataset_out = '/content/drive/MyDrive/Tfg/Dataset/dataset_autoencoder_5000'
os.makedirs(ruta_dataset_out, exist_ok=True)

# objectiu d'imatges per vídeo
OBJECTIU_PER_VIDEO = 1666
LLINDAR_CONFIANCA_YOLO = 0.70 # demanem seguretat per tenir un bon retall

print("Carregant el model YOLO...")
model = YOLO(ruta_yolo_nou)

# funcionament principal
imatges_totals_guardades = 0

for idx_video, ruta_video in enumerate(rutes_videos_in):
    if not os.path.exists(ruta_video):
        print(f"Error: no s'ha trobat el vídeo {ruta_video}, passem al següent.")
        continue

    cap = cv2.VideoCapture(ruta_video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # calculem l'interval de frames per no processar-ho tot
    salt_frames = max(1, total_frames // (OBJECTIU_PER_VIDEO // 10))

    print(f"\nProcessant vídeo {idx_video+1}: {os.path.basename(ruta_video)}")
    print(f"Total frames: {total_frames}. Processarem cada {salt_frames} frames.")

    frames_processats = 0
    imatges_guardades_aquest_video = 0

    while cap.isOpened() and imatges_guardades_aquest_video < OBJECTIU_PER_VIDEO:
        # avancem els frames ràpid
        cap.set(cv2.CAP_PROP_POS_FRAMES, frames_processats)
        ret, frame = cap.read()

        if not ret: break

        # inferència només pel frame actual
        resultats = model(frame, conf=LLINDAR_CONFIANCA_YOLO, verbose=False)

        if resultats[0].boxes is not None and resultats[0].masks is not None:
            boxes = resultats[0].boxes.xyxy.cpu().numpy().astype(int)
            masks_polygons = resultats[0].masks.xy

            for box, polygon in zip(boxes, masks_polygons):
                x1, y1, x2, y2 = box

                # preparem fons blanc i màscara
                fons_blanc = np.ones_like(frame) * 255
                mask_canvas = np.zeros(frame.shape[:2], dtype=np.uint8)
                poly_points = np.array(polygon, dtype=np.int32)
                cv2.fillPoly(mask_canvas, [poly_points], 255)

                # separem el porc i invertim el fons
                porc_aillat = cv2.bitwise_and(frame, frame, mask=mask_canvas)
                mask_invertida = cv2.bitwise_not(mask_canvas)
                fons_amb_forat = cv2.bitwise_and(fons_blanc, fons_blanc, mask=mask_invertida)

                imatge_final_blanca = cv2.add(porc_aillat, fons_amb_forat)
                porc_retallat = imatge_final_blanca[max(0, y1):y2, max(0, x1):x2]

                # si la imatge és vàlida, la redimensionem i guardem
                if porc_retallat.size > 0 and porc_retallat.shape[0] > 20 and porc_retallat.shape[1] > 20:
                    porc_128 = cv2.resize(porc_retallat, (128, 128))

                    nom_arxiu = f"v{idx_video}_f{frames_processats}_p{imatges_guardades_aquest_video}.jpg"
                    ruta_guardat = os.path.join(ruta_dataset_out, nom_arxiu)
                    cv2.imwrite(ruta_guardat, porc_128)

                    imatges_guardades_aquest_video += 1
                    imatges_totals_guardades += 1

        frames_processats += salt_frames

    cap.release()
    print(f"Vídeo {idx_video+1} acabat. S'han guardat {imatges_guardades_aquest_video} imatges.")

print(f"\nDataset creat. Hem guardat un total de {imatges_totals_guardades} imatges a {ruta_dataset_out}.")


## Entrenament AutoEncoder

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# rutes i configuració
ruta_dataset = '/content/drive/MyDrive/Tfg/Dataset/dataset_autoencoder_5000'
ruta_model_sortida = '/content/drive/MyDrive/Tfg/Dataset/encoder_porcs_definitiu.pth'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilitzant dispositiu: {device}")

# hiperparàmetres
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-3
LATENT_DIM = 512 # la mida del vector latent

# preparació de dades
class PorcsDataset(Dataset):
    def __init__(self, ruta_carpeta, transform=None):
        self.rutes_imatges = glob.glob(os.path.join(ruta_carpeta, "*.jpg"))
        self.transform = transform

    def __len__(self):
        return len(self.rutes_imatges)

    def __getitem__(self, idx):
        ruta_img = self.rutes_imatges[idx]
        imatge = Image.open(ruta_img).convert('RGB')
        if self.transform:
            imatge = self.transform(imatge)
        return imatge

transformacions = transforms.Compose([
    transforms.ToTensor(),
])

dataset = PorcsDataset(ruta_dataset, transform=transformacions)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"S'han carregat {len(dataset)} imatges per l'entrenament.")

# definició de l'arquitectura
class AutoEncoder(nn.Module):
    def __init__(self, latent_dim):
        super(AutoEncoder, self).__init__()

        # encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, latent_dim)
        )

        # decoder
        self.decoder_linear = nn.Linear(latent_dim, 256 * 8 * 8)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        latent = self.encoder(x)
        x_recon = self.decoder_linear(latent)
        x_recon = x_recon.view(-1, 256, 8, 8)
        reconstruccio = self.decoder_conv(x_recon)
        return reconstruccio

model = AutoEncoder(LATENT_DIM).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# entrenament
print("\nComencem l'entrenament de l'autoencoder...")

for epoch in range(EPOCHS):
    loss_total = 0.0
    for imatges in dataloader:
        imatges = imatges.to(device)

        optimizer.zero_grad()
        reconstruccions = model(imatges)

        loss = criterion(reconstruccions, imatges)
        loss.backward()
        optimizer.step()

        loss_total += loss.item()

    loss_mitja = loss_total / len(dataloader)
    print(f"Època {epoch+1}/{EPOCHS} - Loss: {loss_mitja:.6f}")

# guardem el model
torch.save(model.encoder.state_dict(), ruta_model_sortida)
print(f"\nEntrenament completat. L'encoder s'ha guardat a {ruta_model_sortida}")


## Execució final + Dashboard

In [ ]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from ultralytics import YOLO
from ultralytics.utils import ROOT
import yaml
import os
import numpy as np
from PIL import Image
from scipy.optimize import linear_sum_assignment
import math

# rutes i paràmetres inicials
ruta_yolo = '/content/drive/MyDrive/Tfg/comparativa_models/yolov8m_definitiu_220imgs/weights/best.pt'
ruta_encoder = '/content/drive/MyDrive/Tfg/Dataset/encoder_porcs_definitiu.pth'
ruta_base = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt'

ruta_video_in = '/content/drive/MyDrive/Tfg/Dataset/dataset_gt/Dataset_nova_granja/Camera2D/Camera2D/room1_cam2_2026-04-19_18-10-06_retallat.mp4'
ruta_video_out = os.path.join(ruta_base, 'Video_test_1/RESULTAT_PANELL_LATERAL_DEBUG.mp4')
os.makedirs(os.path.dirname(ruta_video_out), exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ajustem el tracker
ruta_tracker_original = os.path.join(ROOT, 'cfg', 'trackers', 'botsort.yaml')
with open(ruta_tracker_original, 'r') as f:
    config_tracker = yaml.safe_load(f)
config_tracker['track_buffer'] = 120
ruta_tracker_botsort = '/content/custom_botsort_buf120.yaml'
with open(ruta_tracker_botsort, 'w') as f:
    yaml.dump(config_tracker, f)

# carreguem models
print("Carregant els models de YOLO i Autoencoder...")
model_yolo = YOLO(ruta_yolo)

model_encoder = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(),
    nn.Flatten(),
    nn.Linear(256 * 8 * 8, 512)
)
model_encoder.load_state_dict(torch.load(ruta_encoder, map_location=device))
model_encoder = model_encoder.to(device)
model_encoder.eval()

transform_encoder = transforms.Compose([transforms.ToTensor()])

def extreure_adn_custom(imatge_pil):
    with torch.no_grad():
        tensor = transform_encoder(imatge_pil).unsqueeze(0).to(device)
        return model_encoder(tensor)

# preparació del vídeo i panell d'informació
memoria_identitats = {}
equivalencies_ids = {}
tracker_a_mestre = {}
seguent_id_mestre = 1
LLINDAR_CONFIANCA = 0.40

cap = cv2.VideoCapture(ruta_video_in)
if not cap.isOpened():
    print(f"Error: no es pot obrir el vídeo {ruta_video_in}")

fps = int(cap.get(cv2.CAP_PROP_FPS))
if fps == 0: fps = 30
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

amplada_panell = 450
width_total = width + amplada_panell
out = cv2.VideoWriter(ruta_video_out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width_total, height))

estat_porcs_dashboard = {}
zona_menjadora = (886, 650, 1283, 822)

METRES_AMPLADA_CORRAL = 5.0
PIXELS_PER_METRE = width / METRES_AMPLADA_CORRAL
LLINDAR_MOVIMENT_PX = 7.0

print(f"Iniciant el procés. Tenim uns {total_frames} frames teòrics.")
print(f"Conversió establerta: 1m equival a {PIXELS_PER_METRE:.1f}px")

# procés frame a frame
frames_processats = 0

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        print(f"Lectura finalitzada al frame {frames_processats}")
        break

    frames_processats += 1

    if frames_processats % 50 == 0:
        print(f"Processant frame {frames_processats}...")

    # apliquem el YOLO i tracking
    resultats = model_yolo.track(frame, tracker=ruta_tracker_botsort, conf=0.50, iou=0.30, persist=True, verbose=False)
    frame_dibuixat = frame.copy()

    # preparem el llenç amb vídeo a l'esquerra i panell a la dreta
    canvas = np.zeros((height, width_total, 3), dtype=np.uint8)
    canvas[:, width:] = (30, 30, 30)

    capses_pendents, adns_pendents, boxes_data_per_dibuixar = [], [], []

    if resultats[0].boxes is not None and resultats[0].boxes.id is not None and resultats[0].masks is not None:
        boxes = resultats[0].boxes.xyxy.cpu().numpy().astype(int)
        track_ids = resultats[0].boxes.id.int().cpu().numpy()
        masks_polygons = resultats[0].masks.xy

        identitats_actives_al_frame = set()
        for t_id in track_ids:
            if t_id in equivalencies_ids:
                identitats_actives_al_frame.add(equivalencies_ids[t_id])

        # recollim informació visual
        for box, track_id, polygon in zip(boxes, track_ids, masks_polygons):
            x1, y1, x2, y2 = box
            area_box = (x2 - x1) * (y2 - y1)
            boxes_data_per_dibuixar.append((box, track_id))

            if track_id not in equivalencies_ids and area_box > 6000:
                mask_canvas = np.zeros(frame.shape[:2], dtype=np.uint8)
                cv2.fillPoly(mask_canvas, [np.array(polygon, dtype=np.int32)], 255)
                porc_aillat = cv2.bitwise_and(frame, frame, mask=mask_canvas)
                porc_retallat = porc_aillat[y1:y2, x1:x2]

                if porc_retallat.size > 0:
                    porc_128 = cv2.resize(porc_retallat, (128, 128))
                    adn_actual = extreure_adn_custom(Image.fromarray(cv2.cvtColor(porc_128, cv2.COLOR_BGR2RGB)))
                    capses_pendents.append(track_id)
                    adns_pendents.append(adn_actual)

        # matching de característiques amb l'algorisme hongarès
        if len(capses_pendents) > 0:
            noms_memoria_disponibles = [nom for nom in memoria_identitats.keys() if nom not in identitats_actives_al_frame]
            if len(noms_memoria_disponibles) > 0:
                matriu_similitud = np.zeros((len(capses_pendents), len(noms_memoria_disponibles)))
                for i, adn_pendent in enumerate(adns_pendents):
                    for j, nom_mestre in enumerate(noms_memoria_disponibles):
                        matriu_similitud[i, j] = max(0, F.cosine_similarity(adn_pendent, memoria_identitats[nom_mestre]).item())

                matriu_costos = 1.0 - matriu_similitud
                files_assignades, columnes_assignades = linear_sum_assignment(matriu_costos)

                for fila, columna in zip(files_assignades, columnes_assignades):
                    sim_final = matriu_similitud[fila, columna]
                    t_id = capses_pendents[fila]

                    if sim_final >= LLINDAR_CONFIANCA:
                        m_guanyador = noms_memoria_disponibles[columna]
                        equivalencies_ids[t_id] = m_guanyador
                        tracker_a_mestre[t_id] = m_guanyador
                        memoria_identitats[m_guanyador] = (0.9 * memoria_identitats[m_guanyador]) + (0.1 * adns_pendents[fila])
                        identitats_actives_al_frame.add(m_guanyador)
                    else:
                        nom_nou = f"Ind_{seguent_id_mestre:02d}"
                        equivalencies_ids[t_id] = nom_nou
                        tracker_a_mestre[t_id] = nom_nou
                        memoria_identitats[nom_nou] = adns_pendents[fila]
                        identitats_actives_al_frame.add(nom_nou)
                        seguent_id_mestre += 1
            else:
                for i, t_id in enumerate(capses_pendents):
                    nom_nou = f"Ind_{seguent_id_mestre:02d}"
                    equivalencies_ids[t_id] = nom_nou
                    tracker_a_mestre[t_id] = nom_nou
                    memoria_identitats[nom_nou] = adns_pendents[i]
                    identitats_actives_al_frame.add(nom_nou)
                    seguent_id_mestre += 1

    # càlcul de distàncies i estat
    for box, track_id in boxes_data_per_dibuixar:
        x1, y1, x2, y2 = map(int, box)

        if track_id in equivalencies_ids:
            id_mestre = equivalencies_ids[track_id]
            cx, cy = int(x1 + (x2-x1)/2), int(y1 + (y2-y1)/2)

            if id_mestre not in estat_porcs_dashboard:
                estat_porcs_dashboard[id_mestre] = {
                    'dist_total_px': 0.0,
                    'frames_menjant': 0,
                    'visites_menjadora': 0,
                    'estat': 'Actiu',
                    'ultim_centre': (cx, cy),
                    'a_la_menjadora': False
                }

            dades = estat_porcs_dashboard[id_mestre]

            dx = cx - dades['ultim_centre'][0]
            dy = cy - dades['ultim_centre'][1]
            dist_pixeles = math.sqrt(dx**2 + dy**2)

            # filtre de moviments molt petits
            if dist_pixeles < LLINDAR_MOVIMENT_PX:
                dades['estat'] = 'Repos'
            else:
                dades['estat'] = 'Actiu'
                dades['dist_total_px'] += dist_pixeles

            dades['ultim_centre'] = (cx, cy)

            # comprovem si és a la zona de la menjadora
            zm_x1, zm_y1, zm_x2, zm_y2 = zona_menjadora
            if (zm_x1 < cx < zm_x2) and (zm_y1 < cy < zm_y2):
                dades['frames_menjant'] += 1
                if not dades['a_la_menjadora']:
                    dades['visites_menjadora'] += 1
                    dades['a_la_menjadora'] = True
            else:
                dades['a_la_menjadora'] = False

            # dibuixem la informació sobre el porc al frame
            cv2.rectangle(frame_dibuixat, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.rectangle(frame_dibuixat, (x1, y1-25), (x1+80, y1), (0, 255, 0), -1)
            cv2.putText(frame_dibuixat, id_mestre, (x1+5, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    # marquem la bàscula al vídeo
    cv2.rectangle(frame_dibuixat, (zona_menjadora[0], zona_menjadora[1]), (zona_menjadora[2], zona_menjadora[3]), (255, 0, 0), 2)
    cv2.putText(frame_dibuixat, "BASCULA", (zona_menjadora[0], zona_menjadora[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    canvas[:, :width] = frame_dibuixat

    # informació per al panell lateral
    cv2.putText(canvas, "METRIQUES", (width + 20, 40), cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 2)
    cv2.line(canvas, (width + 20, 55), (width_total - 20, 55), (200, 200, 200), 1)

    y_offset = 90
    for id_porc in sorted(estat_porcs_dashboard.keys()):
        dades_p = estat_porcs_dashboard[id_porc]
        dist_metres = dades_p['dist_total_px'] / PIXELS_PER_METRE
        temps_menjant_segs = dades_p['frames_menjant'] // fps

        color_estat = (0, 255, 0) if dades_p['estat'] == 'Actiu' else (0, 255, 255)

        text_id = f"[{id_porc}] {dades_p['estat']}"
        text_dades = f"Dist: {dist_metres:.1f}m | Bascula: {dades_p['visites_menjadora']}x ({temps_menjant_segs}s)"

        cv2.putText(canvas, text_id, (width + 20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_estat, 1)
        cv2.putText(canvas, text_dades, (width + 20, y_offset + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)
        y_offset += 55

    out.write(canvas)

cap.release()
out.release()
print(f"\nProcés completat. El vídeo s'ha desat a {ruta_video_out}")
